# Metaflora Incubus v1 — free GPU build
This notebook uses a compact <=7.5B QLoRA profile: 4-bit NF4, 2048-token training context and LoRA rank 16. A free T4 is not guaranteed to be available, and 9B training is deliberately rejected. Checkpoints go to Drive or a separate private Hub branch. Public upload remains blocked until the signed eval gates pass.

In [ ]:
import os
import subprocess
import sys

subprocess.run(
    [
        "git",
        "clone",
        "--depth",
        "1",
        "https://github.com/metaflora-app/metaflora-incubus.git",
        "/content/metaflora-incubus",
    ],
    check=True,
)
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--require-hashes",
        "-r",
        "/content/metaflora-incubus/requirements/cloud-linux.lock",
    ],
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--no-deps", "-e", "/content/metaflora-incubus"],
    check=True,
)
os.chdir("/content/metaflora-incubus")

In [ ]:
# Put these values in Colab Secrets; notebook output never prints them.
try:
    from google.colab import userdata

    secret_names = (
        "HF_TOKEN",
        "INCUBUS_CHECKPOINT_HMAC_KEY",
        "INCUBUS_SOURCE_REPO",
        "INCUBUS_SOURCE_REVISION",
        "INCUBUS_DATASET_REPO",
        "INCUBUS_DATASET_REVISION",
        "INCUBUS_DATASET_SHA256",
    )
    for key in secret_names:
        os.environ[key] = userdata.get(key)
except ImportError:
    from kaggle_secrets import UserSecretsClient

    secrets = UserSecretsClient()
    secret_names = (
        "HF_TOKEN",
        "INCUBUS_CHECKPOINT_HMAC_KEY",
        "INCUBUS_SOURCE_REPO",
        "INCUBUS_SOURCE_REVISION",
        "INCUBUS_DATASET_REPO",
        "INCUBUS_DATASET_REVISION",
        "INCUBUS_DATASET_SHA256",
    )
    for key in secret_names:
        os.environ[key] = secrets.get_secret(key)

In [ ]:
# Change the parameter count only to the exact count of the private build input.
command = [
    sys.executable,
    "scripts/run_free_gpu.py",
    "--execute",
    "--run-id",
    "incubus-v1-run",
    "--parameter-count",
    "7000000000",
    "--checkpoint-backend",
    "hf_private_branch",
    "--checkpoint-location",
    "metaflora/incubus-checkpoints",
    "--checkpoint-branch",
    "incubus-training-v1",
]
subprocess.run(command, check=True)